## This notebook is made to cover the practical part of the Machine Learning project that is related to creating a classification prediction model

Before introducing any kind of information in the dataset and/or features, the dataset **MUST BE SPLITTED**

Since we are dealing with a small unbalances dataset 4424 records, the split is going to be 80-20 with K-fold cross-validation to avoid problems with the target column since that is the feature we want to predict. The number of folds we do in the train data could also be changed to see which result we can get out of this parameter, first we are going to start with k = 10

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

raw_dataframe = pd.read_csv('dropout_dataset.csv', sep=';')


#Before really splitting the data, some cleaning is done to avoid annoying errors in the future

for column in raw_dataframe.columns:
    raw_dataframe.rename(columns = {f'{column}':f'{column.rstrip().lstrip()}'})



X = raw_dataframe.drop(columns=["Target"])
y = raw_dataframe["Target"]



X_train, X_test, Y_train, Y_test = train_test_split(X,y,test_size=0.2, random_state=42)


Now that the split have been done, we need to define a strategy to deal with the imbalanced aspect of the dataset. In the reference paper, they used SMOTE, ADASYN and Logistic regression, regarding the avaliable resampling methods, we can try to apply 
- Oversampling techniques
  1. Borderline-SMOTE
  2. random over-sampling
  3. SMOTE-ENN
  4. SMOTENC (especifically designed for nominal and continuous features)

OR

- Algorithmic Solutions
  1. Weighted loss function
  2. Classical weight adjustment
  3. Focal loss


Since our dataset is small, there will be a very big loss of information if we do under-sampling because reducing even more the dataset will pose even more problems, the oversampling techniques will be used for this case

### Create a sample space using borderline SMOTE

In [26]:
from collections import Counter
from imblearn.over_sampling import BorderlineSMOTE

print('Original training dataset shape %s' % Counter(Y_train))

sm = BorderlineSMOTE(random_state=42)
X_borderline_SMOTE, Y_borderline_SMOTE = sm.fit_resample(X_train,Y_train)

print('Resampled training dataset shape %s' % Counter(Y_borderline_SMOTE))


Original training dataset shape Counter({'Graduate': 1791, 'Dropout': 1105, 'Enrolled': 643})
Resampled training dataset shape Counter({'Dropout': 1791, 'Enrolled': 1791, 'Graduate': 1791})


### Create a sample space using random over sampler

In [27]:
from imblearn.over_sampling import RandomOverSampler

print('Original training dataset shape %s' % Counter(Y_train))

ros = RandomOverSampler(random_state=42)
X_random_over_sampler, Y_random_over_sampler = ros.fit_resample(X_train,Y_train)

print('Resampled training dataset shape %s' % Counter(Y_random_over_sampler))


Original training dataset shape Counter({'Graduate': 1791, 'Dropout': 1105, 'Enrolled': 643})
Resampled training dataset shape Counter({'Dropout': 1791, 'Enrolled': 1791, 'Graduate': 1791})


### Create a sample space using random over SMOTE EEN

Here, the result is a little different than those we saw in the other resample stretegies

In [28]:
from imblearn.combine import SMOTEENN

print('Original training dataset shape %s' % Counter(Y_train))

sme = SMOTEENN(random_state=42)
X_smote_een, Y_smote_een = sme.fit_resample(X_train,Y_train)

print('Resampled training dataset shape %s' % Counter(Y_smote_een))


Original training dataset shape Counter({'Graduate': 1791, 'Dropout': 1105, 'Enrolled': 643})
Resampled training dataset shape Counter({'Enrolled': 1154, 'Dropout': 868, 'Graduate': 613})


### Create a sample space using random over SMOTENC

Here, the result is a little different than those we saw in the other resample stretegies

In [29]:
from imblearn.over_sampling import SMOTENC

print('Original training dataset shape %s' % Counter(Y_train))

smnc = SMOTENC(random_state=42, categorical_features= [1,3,5,7])
X_smote_smnc, Y_smote_smnc = smnc.fit_resample(X_train,Y_train)

print('Resampled training dataset shape %s' % Counter(Y_smote_smnc))

Original training dataset shape Counter({'Graduate': 1791, 'Dropout': 1105, 'Enrolled': 643})
Resampled training dataset shape Counter({'Dropout': 1791, 'Enrolled': 1791, 'Graduate': 1791})


Regarding the algorithms to build the models, since our task is to classificate or fit into classification the three categories of students based on the given existing features:
- Success
- Relative success
- Failure

And knowing that in the reference study they used **Logistic regression, SVM, decision tree, random forest, Gradient boosting, Xtreme gradient boosting, legit boost and cat boost**. Some other avaliable options for algorithms are:
- **Probabilistic & Linear Models**
    1. Naïve Bayes (NB)
- **Neural Networks**
    1. Multi-Layer Perceptron (MLP - Feedforward Neural Network)
- **Rule-Based & Distance-Based Models**
    1. K-Nearest Neighbors (KNN)
- **Ensemble & Evolutionary Methods**
    1. Voting classifier
    2. Genetic algorithms

**1. First part of the study: For each model (or one selected model), we can test different sample spaces -> 4 x N_models** 
This will give the output of which sampling method should be used in the study

**2. Second part of the study: for each model, which one can give more satisfactory results -> N_models**
This will give the output of which model is more appropriate for this task

**3. For the most appropriate model, what k-fold is ideal for getting better results(evaluate if it makes sense)**

**4. Compare the results with the study results and draw conclusions**

### Which sample strategy yields the best result?

To answer this question, we will take a similar approach to the reference paper, we pick first a classification method and then run that method with the different sample spaces that we have and then compare the different performances of them.
Taking into consideration our dataset carachteristics and the goal of the model (correctly classify the most under-represented class in the sample set "Partial success/enrolled"), the most appropriated ones are the ones that evaluates how well the classification is done to the most crititcal class, with that in mind, the most appropriate evaluation method is the F1 score (the same one used in the paper). This also keeps the comparision fair so we don't end up in the case of comparing apples to oranges.

The naive bayes method is the one that would require the biggest dataset treatment before applying the model and the KNN is one model that would suffer from the curse of dimensionality, since on this step, the goal is to evaluate which oversampling method is the best, we are going to proceed with the model that presents the most straight forward setup to this evaluation MLP.

To prepare our MLP model, these are the preparations that we need to do:
1. Normalize the continuous features
2. Normalize discrete numerical features
3. One-hot encode the encoded lavel categorical features

#### 1.Normalize continuous features
To normalize them, we will use min-max scaling (we are aware that this causes loss of information)

In [30]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_treatment_test = X_train

df_continuous_columns = ['Previous qualification (grade)','Admission grade', 'Unemployment rate', 
    'Inflation rate','GDP', 'Curricular units 1st sem (grade)','Curricular units 2nd sem (grade)']

df_continuous_columns_data = X_train_treatment_test[df_continuous_columns]

df_continuous_columns_normalized = scaler.fit_transform(df_continuous_columns_data)

df_continuous_columns_normalized = pd.DataFrame(df_continuous_columns_normalized, columns=df_continuous_columns)

X_train_treatment_test[df_continuous_columns] = df_continuous_columns_normalized

print(X_train_treatment_test[df_continuous_columns])

      Previous qualification (grade)  Admission grade  Unemployment rate  \
3383                        0.578947         0.578947           0.372093   
2840                        0.494737         0.340000           1.000000   
564                         0.401053         0.052632           0.732558   
1786                        0.401053         0.324211           0.000000   
3900                             NaN              NaN                NaN   
...                              ...              ...                ...   
3444                        0.315789         0.389474           0.732558   
466                         0.494737         0.432632           1.000000   
3092                        0.368421         0.241053           0.372093   
3772                             NaN              NaN                NaN   
860                         0.368421         0.307368           0.209302   

      Inflation rate       GDP  Curricular units 1st sem (grade)  \
3383        0.48888

As we can see from the output of the print statement, the is a significant amount of NaN values, we will leave the treatment of those after we handle every column to avoid redundant work

#### 2.Normalize discrete numerical features
To normalize them, we will use min-max scaling (we are aware that this causes loss of information)

In [32]:
df_discrete_columns =['Age at enrollment', 'Curricular units 1st sem (credited)','Curricular units 1st sem (enrolled)',
     'Curricular units 1st sem (evaluations)','Curricular units 1st sem (approved)',
     'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (enrolled)',
     'Curricular units 2nd sem (evaluations)','Curricular units 2nd sem (approved)',
     'Curricular units 2nd sem (without evaluations)'
    ]

for column in df_discrete_columns:
    min_val = X_train_treatment_test[column].min()
    max_val = X_train_treatment_test[column].max()
    print(f"Column: {column}")
    print(f"Min: {min_val}, Max: {max_val}")
    print("-" * 50)

Column: Age at enrollment
Min: 17, Max: 70
--------------------------------------------------
Column: Curricular units 1st sem (credited)
Min: 0, Max: 20
--------------------------------------------------
Column: Curricular units 1st sem (enrolled)
Min: 0, Max: 26
--------------------------------------------------
Column: Curricular units 1st sem (evaluations)
Min: 0, Max: 45
--------------------------------------------------
Column: Curricular units 1st sem (approved)
Min: 0, Max: 26
--------------------------------------------------
Column: Curricular units 1st sem (without evaluations)
Min: 0, Max: 12
--------------------------------------------------
Column: Curricular units 2nd sem (enrolled)
Min: 0, Max: 23
--------------------------------------------------
Column: Curricular units 2nd sem (evaluations)
Min: 0, Max: 33
--------------------------------------------------
Column: Curricular units 2nd sem (approved)
Min: 0, Max: 20
--------------------------------------------------
C

Since our discrete columns have a fairly significant range, we will normalize them

In [34]:
discrete_scaler = MinMaxScaler()

df_discrete_columns_data = X_train_treatment_test[df_discrete_columns]

df_discrete_columns_normalized = discrete_scaler.fit_transform(df_discrete_columns_data)

df_discrete_columns_normalized = pd.DataFrame(df_discrete_columns_normalized, columns=df_discrete_columns)

X_train_treatment_test[df_discrete_columns] = df_discrete_columns_normalized

print(X_train_treatment_test[df_discrete_columns])

      Age at enrollment  Curricular units 1st sem (credited)  \
3383           0.113208                                 0.35   
2840           0.075472                                 0.00   
564            0.056604                                 0.00   
1786           0.113208                                 0.50   
3900                NaN                                  NaN   
...                 ...                                  ...   
3444           0.018868                                 0.00   
466            0.132075                                 0.10   
3092           0.245283                                 0.00   
3772                NaN                                  NaN   
860                 NaN                                  NaN   

      Curricular units 1st sem (enrolled)  \
3383                             0.423077   
2840                             0.230769   
564                              0.230769   
1786                             0.576923   
3900  